# Laboratorio Evaluado: WBCD y Evaluación Multimodelo

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD contiene radios y perímetros de células. Debes entrenar k-NN y Árboles de Decisión, y validar qué pasa si no escalas los datos.

## 1. Carga de Datos y Exploración

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score

# Nombres de columnas del WBCD: id, diagnosis y 30 medidas (10 características x [promedio, error estándar, peor valor])
base = ['radius','texture','perimeter','area','smoothness','compactness','concavity','concave_points','symmetry','fractal_dimension']
columnas = ['id','diagnosis'] + [f'{b}_{s}' for s in ['mean','se','worst'] for b in base]

try:
    df_wbcd = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data", header=None, names=columnas)
    print("Datos cargados desde UCI.")
except Exception:
    # Si no hay acceso a UCI, scikit-learn trae exactamente la misma base WBCD
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer()
    df_wbcd = pd.DataFrame(d.data, columns=columnas[2:])
    df_wbcd.insert(0, 'diagnosis', np.where(d.target == 0, 'M', 'B'))
    df_wbcd.insert(0, 'id', range(len(df_wbcd)))
    print("UCI no disponible: se cargó la misma base WBCD desde scikit-learn.")

print(df_wbcd.shape)
print(df_wbcd['diagnosis'].value_counts())
print(df_wbcd.isnull().sum().sum(), "valores nulos")
df_wbcd.head()

## 2. Preprocesamiento

In [ ]:
# M (Maligno) = 1, B (Benigno) = 0
df_wbcd['diagnosis'] = df_wbcd['diagnosis'].map({'M': 1, 'B': 0})

# Separar features y target (el id no aporta información)
X = df_wbcd.drop(columns=['id', 'diagnosis'])
y = df_wbcd['diagnosis']

# 80% entrenamiento / 20% prueba (stratify mantiene la proporción de malignos)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 3. Escalamiento (Solo para k-NN)

In [ ]:
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)  # fit solo con train
X_test_esc = scaler.transform(X_test)        # test solo se transforma (evita fuga de datos)
print("Datos escalados.")

## 4. Modelado y Matrices de Confusión

In [ ]:
# Modelos entrenados con y sin escalar para validar el efecto del escalamiento
knn_esc = KNeighborsClassifier(n_neighbors=5).fit(X_train_esc, y_train)
knn_crudo = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
arbol = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
arbol_esc = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train_esc, y_train)

modelos = {
    'k-NN CON escalar':  knn_esc.predict(X_test_esc),
    'k-NN SIN escalar':  knn_crudo.predict(X_test),
    'Árbol SIN escalar': arbol.predict(X_test),
    'Árbol CON escalar': arbol_esc.predict(X_test_esc),
}

# Formato de la matriz: [[VN, FP], [FN, VP]]  (fila = real, columna = predicho)
for nombre, pred in modelos.items():
    cm = confusion_matrix(y_test, pred)
    print(f"\n=== {nombre} ===")
    print(cm)
    print(f"Exactitud: {accuracy_score(y_test, pred)*100:.2f}% | Falsos Negativos: {cm[1,0]} | Falsos Positivos: {cm[0,1]} | Recall (maligno): {recall_score(y_test, pred)*100:.2f}%")

**Lectura de resultados (split 80/20, `random_state=42`, 114 pacientes de prueba, 42 malignos):**

| Modelo | Exactitud | Falsos Negativos | Falsos Positivos |
|---|---|---|---|
| k-NN con escalar | 95.61% | 4 | 1 |
| k-NN sin escalar | 91.23% | 9 | 1 |
| Árbol sin escalar | 91.23% | 8 | 2 |
| Árbol con escalar | 91.23% | 8 | 2 |

- Sin escalar, k-NN pasa de 4 a 9 falsos negativos: variables como `area` (cientos o miles) dominan la distancia sobre variables como `smoothness` (~0.1).
- El Árbol da **exactamente la misma matriz** con y sin escalar.

## Evaluación Teórica / Decisión de Negocios

1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?
2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?

*Escribe tu respuesta aquí:*

**1. Error Tipo II (Falso Negativo).** Es cuando el modelo dice "benigno" y el tumor en realidad es **maligno**. La paciente recibe un diagnóstico tranquilizador, no inicia tratamiento y el cáncer sigue avanzando; cuando se detecta de nuevo, suele estar en una etapa más grave y con menos opciones de curación. Por eso cuesta vidas. El Falso Positivo (Error Tipo I) en cambio genera angustia y estudios adicionales (como una biopsia), pero la paciente sigue recibiendo atención. En este problema es preferible tolerar más falsos positivos si con eso bajan los falsos negativos, por lo que la métrica clave no es solo la exactitud sino el **Recall** de la clase maligna. En nuestros resultados, k-NN sin escalar dejó pasar 9 casos malignos frente a 4 con escalar: el mismo modelo, mal preparado, habría dejado sin tratamiento a más del doble de pacientes.

**2. Por qué el Árbol no se afectó por la escala.** El árbol no calcula distancias: en cada nodo hace una pregunta de umbral del tipo `area_worst <= 880`, y busca el corte que mejor separa las clases. Escalar es una transformación que conserva el orden de los valores, así que el mejor corte separa a exactamente los mismos pacientes, solo cambia el número del umbral. Por eso las matrices del Árbol con y sin escalar son idénticas (8 FN y 2 FP en ambas). k-NN, en cambio, suma las diferencias de todas las columnas, y las de mayor magnitud (`area`, `perimeter`) tapan a las demás. Ojo: que el árbol sea inmune a la escala no lo hace mejor: aquí tuvo 8 falsos negativos y k-NN escalado solo 4.